<a href="https://colab.research.google.com/github/BrayanAQ/Analisis-del-estres-acad-mico-en-estudiantes-universitarios-de-Bangladesh-mediante-Python-en-el-2025/blob/main/Funciones_proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import matplotlib.pyplot as plt
import pandas as pd
from prettytable import PrettyTable


In [37]:
datos = pd.read_csv('/content/Stress Indicators Dataset for Mental Health Classification.csv')


In [35]:
import pandas as pd

def distribucion_riesgo(df, umbral=4):

    # Excluir columnas no indicadoras
    exclude_cols = ['gender', 'age', 'stress_type']
    indicator_cols = [col for col in df.columns if col not in exclude_cols]

    # Calcular puntaje de riesgo (sin modificar el DataFrame original)
    risk_score = (df[indicator_cols] >= umbral).sum(axis=1)

    # Obtener distribución
    total_personas = len(df)
    distribucion = risk_score.value_counts().sort_index()

    # Crear tabla
    print("\n" + "-"*60)
    print("        DISTRIBUCIÓN DE PUNTAJES DE RIESGO")
    print("-"*60)

    table = PrettyTable()
    table.field_names = ["Síntomas graves", "Personas", "Porcentaje", "Porcentaje acumulado"]
    table.align["Síntomas graves"] = "c"
    table.align["Personas"] = "r"
    table.align["Porcentaje"] = "r"
    table.align["Porcentaje acumulado"] = "r"

    acumulado = 0
    for score, count in distribucion.items():
        acumulado += count
        porcentaje = (count / total_personas) * 100
        porcentaje_acum = (acumulado / total_personas) * 100
        table.add_row([f"{score}", f"{count}", f"{porcentaje:.1f}%", f"{porcentaje_acum:.1f}%"])

    print(table)
    print("\n" + "="*60 + "\n")

    # Estadísticas adicionales útiles
    print(f"📊 Resumen rápido:")
    print(f"   • Personas con 0 síntomas graves: {distribucion.get(0, 0)}")
    print(f"   • Personas con ≥5 síntomas graves: {(risk_score >= 5).sum()}")
    print(f"   • Personas con ≥10 síntomas graves: {(risk_score >= 10).sum()}")

distribucion_riesgo(datos, umbral=4)




------------------------------------------------------------
        DISTRIBUCIÓN DE PUNTAJES DE RIESGO
------------------------------------------------------------
+-----------------+----------+------------+----------------------+
| Síntomas graves | Personas | Porcentaje | Porcentaje acumulado |
+-----------------+----------+------------+----------------------+
|        0        |       23 |       1.1% |                 1.1% |
|        1        |       81 |       4.0% |                 5.2% |
|        2        |      172 |       8.6% |                13.8% |
|        3        |      252 |      12.6% |                26.4% |
|        4        |      245 |      12.2% |                38.6% |
|        5        |      315 |      15.8% |                54.4% |
|        6        |      237 |      11.8% |                66.2% |
|        7        |      204 |      10.2% |                76.4% |
|        8        |      158 |       7.9% |                84.4% |
|        9        |       76 |

In [ ]:
def analisis_estres_por_edad(datos, umbral_grave=4):

    df = datos.copy()

    # Crear grupos de edad
    def grupo_edad(edad):
        if edad <= 18:
            return 'Adolescente (≤18)'
        elif edad <= 25:
            return 'Adulto joven (19-25)'
        elif edad <= 35:
            return 'Adulto (26-35)'
        else:
            return 'Adulto mayor (36+)'

    df['grupo_edad'] = df['age'].apply(grupo_edad)

    # Excluir columnas no indicadoras
    exclude_cols = ['gender', 'age', 'stress_type', 'grupo_edad']
    indicator_cols = [col for col in df.columns if col not in exclude_cols]

    # Calcular puntaje de riesgo por persona
    df['risk_score'] = (df[indicator_cols] >= umbral_grave).sum(axis=1)

    # Orden de grupos
    orden_grupos = ['Adolescente (≤18)', 'Adulto joven (19-25)', 'Adulto (26-35)', 'Adulto mayor (36+)']

    print("\n" + "="*85)
    print("                     ANÁLISIS DE GRAVEDAD DE ESTRÉS POR EDAD")
    print("="*85)

    tabla = PrettyTable()
    tabla.field_names = ["Grupo de edad", "Personas", "% Angustia (stress_type=0)",
                         "% Eustrés (stress_type=1)", "% Mixto (stress_type=2)",
                         "Riesgo promedio", "Síntomas graves (media)"]
    tabla.align["Grupo de edad"] = "l"
    tabla.align["Personas"] = "r"
    tabla.align["% Angustia (stress_type=0)"] = "r"
    tabla.align["% Eustrés (stress_type=1)"] = "r"
    tabla.align["% Mixto (stress_type=2)"] = "r"
    tabla.align["Riesgo promedio"] = "r"
    tabla.align["Síntomas graves (media)"] = "r"

    for grupo in orden_grupos:
        if grupo in df['grupo_edad'].values:
            subset = df[df['grupo_edad'] == grupo]
            total = len(subset)

            # Calcular porcentajes de cada tipo de estrés
            pct_angustia = (subset['stress_type'] == 0).sum() / total * 100
            pct_eustres = (subset['stress_type'] == 1).sum() / total * 100
            pct_mixto = (subset['stress_type'] == 2).sum() / total * 100

            riesgo_prom = subset['risk_score'].mean()
            sintomas_prom = subset[indicator_cols].mean().mean()

            tabla.add_row([
                grupo,
                total,
                f"{pct_angustia:.1f}%",
                f"{pct_eustres:.1f}%",
                f"{pct_mixto:.1f}%",
                f"{riesgo_prom:.1f}",
                f"{sintomas_prom:.2f}"
            ])

    print(tabla)

    # Resumen ejecutivo (enfocado en ANGUSTIA, que es la que importa)
    print("\n" + "-"*85)
    print("RESUMEN EJECUTIVO (ENFOQUE EN ANGUSTIA - ESTRÉS NEGATIVO):")
    print("-"*85)

    # Calcular angustia por grupo
    angustia_por_grupo = df.groupby('grupo_edad').apply(lambda x: (x['stress_type'] == 0).mean() * 100)

    # Grupo con mayor angustia
    grupo_max_angustia = angustia_por_grupo.idxmax()
    max_angustia = angustia_por_grupo.max()

    # Grupo con menor angustia
    grupo_min_angustia = angustia_por_grupo.idxmin()
    min_angustia = angustia_por_grupo.min()

    print(f"⚠️  Grupo con MAYOR porcentaje de ANGUSTIA (estrés negativo): {grupo_max_angustia}")
    print(f"   → {max_angustia:.1f}% de este grupo tiene estrés negativo que requiere intervención")

    print(f"\n✓ Grupo con MENOR porcentaje de ANGUSTIA: {grupo_min_angustia}")
    print(f"   → {min_angustia:.1f}% de este grupo tiene estrés negativo")

    # Diferencia
    diferencia = max_angustia - min_angustia
    print(f"\n📊 Diferencia entre grupos: {diferencia:.1f} puntos porcentuales")

    # Grupo con mayor riesgo promedio
    riesgo_por_grupo = df.groupby('grupo_edad')['risk_score'].mean()
    grupo_max_riesgo = riesgo_por_grupo.idxmax()

    print(f"\n⚠️  Grupo con MAYOR número de síntomas graves: {grupo_max_riesgo}")
    print(f"   → Promedio de {riesgo_por_grupo.max():.1f} síntomas graves por persona")

    print("="*85 + "\n")

    return tabla


analisis_estres_por_edad(datos, umbral_grave=4)



                     ANÁLISIS DE GRAVEDAD DE ESTRÉS POR EDAD
+----------------------+----------+----------------------------+---------------------------+-------------------------+-----------------+-------------------------+
| Grupo de edad        | Personas | % Angustia (stress_type=0) | % Eustrés (stress_type=1) | % Mixto (stress_type=2) | Riesgo promedio | Síntomas graves (media) |
+----------------------+----------+----------------------------+---------------------------+-------------------------+-----------------+-------------------------+
| Adolescente (≤18)    |      344 |                       2.9% |                     86.9% |                   10.2% |             5.5 |                    2.55 |
| Adulto joven (19-25) |     1606 |                       3.6% |                     92.4% |                    4.0% |             5.7 |                    2.63 |
| Adulto (26-35)       |       21 |                       0.0% |                    100.0% |                    0.0% |     

/tmp/ipykernel_31414/152104807.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  angustia_por_grupo = df.groupby('grupo_edad').apply(lambda x: (x['stress_type'] == 0).mean() * 100)


Grupo de edad,Personas,% Angustia (stress_type=0),% Eustrés (stress_type=1),% Mixto (stress_type=2),Riesgo promedio,Síntomas graves (media)
Adolescente (≤18),344,2.9%,86.9%,10.2%,5.5,2.55
Adulto joven (19-25),1606,3.6%,92.4%,4.0%,5.7,2.63
Adulto (26-35),21,0.0%,100.0%,0.0%,5.7,2.50
Adulto mayor (36+),29,6.9%,82.8%,10.3%,6.8,2.78


In [ ]:
def top_sintomas_graves(df, umbral=4, top_n=5):

    # Excluir columnas no indicadoras
    exclude_cols = ['gender', 'age', 'stress_type']
    indicator_cols = [col for col in df.columns if col not in exclude_cols]

    # Calcular frecuencia de síntomas graves
    total_personas = len(df)
    frecuencias = {}

    for col in indicator_cols:
        frecuencias[col] = (df[col] >= umbral).sum()

    # Ordenar y tomar top_n
    top_sintomas = sorted(frecuencias.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # Crear tabla
    print("\n" + "-"*60)
    print(f"        TOP {top_n} SÍNTOMAS GRAVES MÁS FRECUENTES (>= {umbral})")
    print("-"*60)

    table = PrettyTable()
    table.field_names = ["#", "Síntoma", "Personas afectadas", "Porcentaje"]
    table.align["#"] = "c"
    table.align["Síntoma"] = "l"
    table.align["Personas afectadas"] = "r"
    table.align["Porcentaje"] = "r"

    for i, (sintoma, count) in enumerate(top_sintomas, 1):
        porcentaje = (count / total_personas) * 100
        # Limpiar nombre del síntoma para mejor visualización
        nombre_limpio = sintoma.replace('_', ' ').title()
        table.add_row([f"{i}", nombre_limpio, f"{count}", f"{porcentaje:.1f}%"])

    print(table)

    # Estadísticas adicionales
    print("\n" + "-"*40)
    print(f"📊 Datos complementarios:")
    print(f"   • Total de personas evaluadas: {total_personas}")
    print(f"   • Total de síntomas analizados: {len(indicator_cols)}")
    print(f"   • Síntoma más frecuente: {top_sintomas[0][0].replace('_', ' ').title()} ({top_sintomas[0][1]} personas)")

    # Mostrar si hay síntomas con 0 casos
    sintomas_cero = [s for s, c in frecuencias.items() if c == 0]
    if sintomas_cero:
        print(f"   • Síntomas sin casos graves: {len(sintomas_cero)}")

    print("="*60 + "\n")

    return top_sintomas  # Opcional: retorna la lista para usarla después



top_sintomas_graves(datos, umbral=4, top_n=5)


------------------------------------------------------------
        TOP 5 SÍNTOMAS GRAVES MÁS FRECUENTES (>= 4)
------------------------------------------------------------
+---+--------------------+--------------------+------------+
| # | Síntoma            | Personas afectadas | Porcentaje |
+---+--------------------+--------------------+------------+
| 1 | Class Attendance   |                917 |      45.9% |
| 2 | Stress Experience  |                651 |      32.6% |
| 3 | Sleep Problems     |                590 |      29.5% |
| 4 | Irritability       |                587 |      29.3% |
| 5 | Academic Conflicts |                581 |      29.0% |
+---+--------------------+--------------------+------------+

----------------------------------------
📊 Datos complementarios:
   • Total de personas evaluadas: 2000
   • Total de síntomas analizados: 23
   • Síntoma más frecuente: Class Attendance (917 personas)



[('class_attendance', np.int64(917)),
 ('stress_experience', np.int64(651)),
 ('sleep_problems', np.int64(590)),
 ('irritability', np.int64(587)),
 ('academic_conflicts', np.int64(581))]